# Uso del Método KNN con scikit-learn

## Introducción a KNN
El método K-Nearest Neighbors (KNN) es un algoritmo de aprendizaje supervisado utilizado para clasificación y regresión. Funciona clasificando un punto nuevo basado en la mayoría de los puntos más cercanos en el espacio de características.

### Parámetros importantes de KNN
- **n_neighbors**: Número de vecinos a considerar.
- **weights**: Función de peso aplicada a los vecinos. Puede ser 'uniform' (todos los vecinos tienen el mismo peso) o 'distance' (los vecinos más cercanos tienen más peso).
- **metric**: La distancia utilizada para calcular la proximidad. Puede ser 'euclidean', 'manhattan', etc. Se pueden definir métricas personalizadas.
- **algorithm**: Algoritmo utilizado para encontrar los vecinos más cercanos. Opciones incluyen 'auto', 'ball_tree', 'kd_tree', y 'brute'.

    - * `brute`: Este es el enfoque más sencillo y directo para KNN. El algoritmo simplemente calcula la distancia entre el punto de consulta y todos los puntos del conjunto de datos de entrenamiento. Una vez calculadas todas las distancias, se ordenan los puntos en función de su distancia al punto de consulta, y los k puntos más cercanos se seleccionan como vecinos.

    - * `kd-tree`: Un KD-Tree (abreviación de "K-dimensional tree") es una estructura de datos en forma de árbol utilizada para organizar puntos en un espacio K-dimensional. Facilita la búsqueda eficiente de los puntos más cercanos, dividiendo recursivamente el espacio en regiones.

    - * `Ball Tree`: Un Ball Tree es otra estructura de datos en forma de árbol utilizada para mejorar la eficiencia en la búsqueda de vecinos más cercanos. En lugar de dividir el espacio en hiperplanos (como lo hace el KD-Tree), el Ball Tree organiza los puntos dentro de bolas (esferas) que contienen subconjuntos de puntos. El Ball Tree es más eficiente que el KD-Tree en situaciones donde los datos tienen muchas dimensiones, pero aún es vulnerable a la maldición de la dimensionalidad en casos extremos.

    - * `auto`: Selecciona automáticamente el mejor algoritmo basado en la naturaleza de los datos.




## Ejemplo 1: Clasificación con el Conjunto de Datos Iris
Primero, importamos las librerías necesarias y el conjunto de datos Iris.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Cargar el conjunto de datos Iris
iris = load_iris()
X = iris.data
y = iris.target

### Dividir el conjunto de datos
Dividimos los datos en conjuntos de entrenamiento y prueba.

In [2]:
# Dividir los datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)


### Entrenar el modelo KNN
Creamos y ajustamos el modelo KNN.

In [3]:
# Crear el clasificador KNN
k = 3  # Número de vecinos
knn_1 = KNeighborsClassifier(n_neighbors=k)
knn_2 = KNeighborsClassifier(n_neighbors=k, metric = 'manhattan', algorithm='kd_tree')
knn_1.fit(X_train, y_train)
knn_2.fit(X_train, y_train)

KNeighborsClassifier(algorithm='kd_tree', metric='manhattan', n_neighbors=3)

### Realizar predicciones
Hacemos predicciones sobre el conjunto de prueba.

In [4]:
# Realizar predicciones
y_pred_1 = knn_1.predict(X_test)
y_pred_2 = knn_2.predict(X_test)

### Evaluar el modelo
Calculamos la precisión y mostramos el informe de clasificación.

In [5]:
# Evaluar el modelo
accuracy = accuracy_score(y_test, y_pred_1)
print(f'Precisión: {accuracy:.2f}')
print(classification_report(y_test, y_pred_1))

accuracy = accuracy_score(y_test, y_pred_2)
print(f'Precisión: {accuracy:.2f}')
print(classification_report(y_test, y_pred_2))


Precisión: 0.97
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        29
           1       0.92      1.00      0.96        23
           2       1.00      0.91      0.95        23

    accuracy                           0.97        75
   macro avg       0.97      0.97      0.97        75
weighted avg       0.98      0.97      0.97        75

Precisión: 0.99
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        29
           1       0.96      1.00      0.98        23
           2       1.00      0.96      0.98        23

    accuracy                           0.99        75
   macro avg       0.99      0.99      0.99        75
weighted avg       0.99      0.99      0.99        75



## Ejemplo 2: KNN con Datos Numéricos y Categóricos
Ahora, vamos a crear un conjunto de datos que contenga tanto datos numéricos como categóricos.
Vamos a resolverlo de dos formas:

* Codificando las variables categoricas con `one-hot-encoding` y aplicando una métrica Minkowski
* Usando la **distancia de Gower** para manejar variables mixtas (numéricas y categóricas). 

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

### Conjunto de datos de ejemplo:
Creamos un pequeño conjunto de datos con dos variables categóricas (por ejemplo, "color" y "forma") y tres variables numéricas (por ejemplo, "altura", "peso" y "edad").

In [7]:
np.random.seed(42)  # Para reproducibilidad

# Crear 100 muestras
n_samples = 100

# Variables categóricas
colors = np.random.choice(['rojo', 'azul', 'verde'], size=n_samples)
shapes = np.random.choice(['circular', 'cuadrado', 'triangular'], size=n_samples)

# Variables numéricas
heights = np.random.uniform(1.5, 2.0, size=n_samples)  # Altura entre 1.5 y 2.0
weights = np.random.uniform(50, 100, size=n_samples)    # Peso entre 50 y 100
ages = np.random.randint(18, 65, size=n_samples)        # Edad entre 18 y 65

# Crear el DataFrame
df = pd.DataFrame({
    'color': colors,
    'forma': shapes,
    'altura': heights,
    'peso': weights,
    'edad': ages,
    'clase': np.random.choice([0, 1], size=n_samples)  # Etiquetas de clase 0 o 1
})


### KNN con `one-hot-encoding`

In [8]:
from sklearn.preprocessing import OneHotEncoder

# Listar las variables categóricas
categorical_cols = ['color', 'forma']

# Separar las variables predictoras y la variable objetivo
X = df.drop('clase', axis=1)
y = df['clase']

# Aplicar One-Hot Encoding solo a las columnas categóricas
one_hot_encoder = OneHotEncoder(sparse_output=False, drop='first')  # 'drop=first' evita colinealidad
X_encoded = one_hot_encoder.fit_transform(X[categorical_cols])

# Unir las variables numéricas con las categóricas codificadas
X_numeric = X[['altura', 'peso', 'edad']].values
X_preprocessed = np.hstack((X_encoded, X_numeric))

# Dividir en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.4, random_state=42)

In [9]:
# Definir el modelo KNN con 3 vecinos
knn = KNeighborsClassifier(n_neighbors=3, metric='euclidean')

# Entrenar el modelo
knn.fit(X_train, y_train)

KNeighborsClassifier(metric='euclidean', n_neighbors=3)

In [10]:
# Hacer predicciones en el conjunto de prueba
y_pred = knn.predict(X_test)

# Calcular y mostrar la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Mostrar un informe de clasificación detallado
print(classification_report(y_test, y_pred))

Accuracy: 0.50
              precision    recall  f1-score   support

           0       0.58      0.62      0.60        24
           1       0.36      0.31      0.33        16

    accuracy                           0.50        40
   macro avg       0.47      0.47      0.47        40
weighted avg       0.49      0.50      0.49        40



### KNN con distancia de `Gower`

In [11]:
def gower_distance(X, Y, categorical_cols):
    """
    Calcula la distancia de Gower entre dos vectores X e Y, que deben ser de la misma longitud.
    categorical_cols: lista de índices de columnas categóricas
    """
    n_columns = len(X)
    gower_dist = np.zeros(n_columns)

    for i in range(n_columns):
        if i in categorical_cols:  # Características categóricas codificadas
            gower_dist[i] = 0 if X[i] == Y[i] else 1
        else:  # Características numéricas
            range_i = max(X_train.iloc[:, i]) - min(X_train.iloc[:, i])
            gower_dist[i] = np.abs(X[i] - Y[i]) / range_i if range_i > 0 else 0

    return np.mean(gower_dist)

def custom_gower_metric(x, y):
    """
    Función personalizada para la métrica de Gower, adaptada para el uso en KNN.
    x: una muestra de prueba
    y: una muestra de entrenamiento
    """
    return gower_distance(x, y, categorical_col_indices)

In [12]:
# Mapeo para color y forma
mappings = {
    'color': {'rojo': 0, 'azul': 1, 'verde': 2},
    'forma': {'circular': 0, 'cuadrado': 1, 'triangular': 2}
}

# Reemplazar los valores categóricos por numéricos
for col in mappings:
    df[col] = df[col].replace(mappings[col])

# Asegúrate de que las columnas categóricas están en X
categorical_cols = ['color', 'forma']

# Separar las variables predictoras y la variable objetivo
X = df.drop('clase', axis=1)
y = df['clase']

# Obtener los índices de las columnas categóricas en X
categorical_col_indices = [X.columns.get_loc(col) for col in categorical_cols if col in X.columns]

# Dividir los datos en entrenamiento y prueba (40% de test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)


In [13]:
# Crear el clasificador KNN con la distancia de Gower personalizada
knn_gower = KNeighborsClassifier(n_neighbors=5, metric=custom_gower_metric)

# Ajustar el modelo
knn_gower.fit(X_train.values, y_train.values)


KNeighborsClassifier(metric=<function custom_gower_metric at 0x000002357C2604C0>)

In [ ]:
# Realizar predicciones en el conjunto de prueba
y_pred = knn_gower.predict(X_test.values)

# Evaluar el modelo
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.45
              precision    recall  f1-score   support

           0       0.55      0.46      0.50        24
           1       0.35      0.44      0.39        16

    accuracy                           0.45        40
   macro avg       0.45      0.45      0.44        40
weighted avg       0.47      0.45      0.46        40



: 

## Ejemplo 3: Impacto de normalizar y estandarizar en KNN.

Es recomendable normalizar o estandarizar las variables en k-NN porque el algoritmo basa su decisión en una distancia en el espacio de características.
Si las variables están en escalas distintas, la distancia queda dominada por la variable con mayor rango.

#### Impacto del Escalado en KNN (Wine Dataset)

En este experimento analizamos cómo influye el escalado de variables en el rendimiento del algoritmo **k-Nearest Neighbors (KNN)** utilizando el dataset `load_wine()` de `sklearn`.

Comparar la precisión del modelo bajo tres escenarios:

1. **Sin escalado**
2. **Normalización (MinMaxScaler)**
3. **Estandarización (StandardScaler)**

Procedimiento:
1. Se carga el dataset de vinos.
2. Se divide en entrenamiento (80%) y test (20%).
3. Se entrena un modelo `KNeighborsClassifier` con:
   - \( k = 5 \)
   - Métrica por defecto (Minkowski con \( p=2 \), es decir, Euclídea).
4. Se evalúa la precisión (`accuracy`) en cada caso.
5. Se representa gráficamente la comparación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 1. Cargar el dataset de vinos
wine = load_wine()
X, y = wine.data, wine.target

# 2. Dividir en train y test (80%-20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Aplicar KNN sin escalado
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
accuracy_no_scaling = accuracy_score(y_test, y_pred)
print(f" Precisión SIN escalado: {accuracy_no_scaling:.2f}")

# 4. Aplicar Normalización (MinMaxScaler)
scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)

knn.fit(X_train_minmax, y_train)
y_pred_minmax = knn.predict(X_test_minmax)
accuracy_minmax = accuracy_score(y_test, y_pred_minmax)
print(f" Precisión con Normalización: {accuracy_minmax:.2f}")

# 5. Aplicar Estandarización (StandardScaler)
scaler_standard = StandardScaler()
X_train_standard = scaler_standard.fit_transform(X_train)
X_test_standard = scaler_standard.transform(X_test)

knn.fit(X_train_standard, y_train)
y_pred_standard = knn.predict(X_test_standard)
accuracy_standard = accuracy_score(y_test, y_pred_standard)
print(f"Precisión con Estandarización: {accuracy_standard:.2f}")

# 6. Comparación visual
scaling_methods = ["Sin Escalar", "Normalización", "Estandarización"]
accuracies = [accuracy_no_scaling, accuracy_minmax, accuracy_standard]

plt.bar(scaling_methods, accuracies, color=['red', 'green', 'blue'])
plt.xlabel("Método de Escalado")
plt.ylabel("Precisión del Modelo")
plt.title("Impacto del Escalado en KNN")
plt.ylim(0.5, 1)
plt.show()


#### Influencia de la Escala en la Frontera de Decisión de KNN


En este experimento se construye un conjunto de datos sintético con dos variables de entrada de escalas muy diferentes:

- **X1**: valores enteros entre 0 y 1000  
- **X2**: valores reales entre 0 y 1  

El objetivo es visualizar cómo afecta esta diferencia de escala al comportamiento geométrico de KNN.


Diseño del experimento

1. Se generan 100 muestras con dos características:
   - Una variable con magnitudes grandes.
   - Otra variable con magnitudes pequeñas.
2. Se asignan etiquetas binarias aleatorias.
3. Se entrena un modelo KNN con:
   - \( k = 5 \)
   - Distancia por defecto (Minkowski con \( p=2 \), es decir, Euclídea).
4. Se representa la frontera de decisión:
   - Sin normalización.
   - Con normalización Min–Max.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

# 1. Generar datos sintéticos
np.random.seed(42)
X1 = np.random.randint(0, 1000, 100)  # Feature con valores grandes
X2 = np.random.rand(100)  # Feature con valores pequeños
y = np.random.choice([0, 1], size=100)  # Dos clases (0 y 1)

X = np.column_stack((X1, X2))  # Juntar en una matriz
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Aplicar KNN sin normalizar
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)

# 3. Normalizar los datos
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Aplicar KNN con datos normalizados
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)

# 5. Visualizar los datos y vecinos cercanos
def plot_knn_decision_boundary(model, X, y, title, xlim, ylim):
    x_min, x_max = xlim
    y_min, y_max = ylim
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10 if x_max > 1 else 6, 5))  # Más ancho si X1 es grande
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolor="k")
    plt.title(title)
    plt.xlabel("X1")
    plt.ylabel("X2")
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.show()

#  Plot sin normalizar (X1 grande, X2 pequeña)
plot_knn_decision_boundary(knn_raw, X_train, y_train, "KNN sin Normalización",
                           xlim=(X_train[:, 0].min(), X_train[:, 0].max()),
                           ylim=(X_train[:, 1].min(), X_train[:, 1].max()))

# Plot con normalización (ambos entre 0 y 1)
plot_knn_decision_boundary(knn_scaled, X_train_scaled, y_train, "KNN con Normalización",
                           xlim=(0, 1), ylim=(0, 1))


## Ejemplo 4: KNN y Clases Desbalanceadas.

#### Impacto de SMOTE en la Frontera de Decisión

En este experimento analizamos cómo afecta el desbalanceo de clases al comportamiento geométrico de KNN y cómo cambia la frontera de decisión tras aplicar **SMOTE**.


1. Se generan datos sintéticos en 2D:
   - **Clase mayoritaria (0)**: 120 puntos alrededor de (1,1).
   - **Clase minoritaria (1)**: 15 puntos alrededor de (0,0).
   - Distribución aproximada: 90% vs 10%.

2. Se divide el conjunto en train y test.

3. Se entrena un modelo:
   - `KNeighborsClassifier`
   - \( k = 9 \)
   - Sin balancear los datos.

4. Se visualiza la frontera de decisión.

5. Se aplica **SMOTE** (Synthetic Minority Over-sampling Technique) sobre el conjunto de entrenamiento.

6. Se reentrena KNN (k=7) y se vuelve a visualizar la frontera.

In [ ]:
! pip install imblearn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE

#  1. Generar datos desbalanceados
np.random.seed(42)
class_major = np.random.randn(120, 2) * 0.8 + [1, 1]  # 90 puntos alrededor de (2,2)
class_minor = np.random.randn(15, 2) * 0.6 + [0, 0]  # 10 puntos alrededor de (0,0)

X = np.vstack((class_major, class_minor))
y = np.array([0] * 120 + [1] * 15)  # Clase 0 (90%) y Clase 1 (10%)

#  2. Dividir en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=422)

# 3. Aplicar KNN con k=7 (antes de balancear)
knn = KNeighborsClassifier(n_neighbors=9)
knn.fit(X_train, y_train)

# 4. Visualizar clasificación KNN antes de balancear
def plot_knn_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolor="k")
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

#  Antes de balancear
plot_knn_decision_boundary(knn, X_train, y_train, "KNN sin Data Augmentation")

y_pred_dist = knn.predict(X_test)
print("=== KNN weights='distance' (k=9) ===")
print(classification_report(y_test, y_pred_dist, digits=3))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_dist)
plt.title("Matriz de confusión - weights='distance'")
plt.show()

#  5. Aplicar SMOTE para balancear clases
smote = SMOTE(sampling_strategy="auto", random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

#  6. Aplicar KNN después de balancear
knn_bal = KNeighborsClassifier(n_neighbors=7)
knn_bal.fit(X_train_bal, y_train_bal)

#  Después de balancear
plot_knn_decision_boundary(knn_bal, X_train_bal, y_train_bal, "KNN con SMOTE (Clases Balanceadas)")

y_pred_dist = knn_bal.predict(X_test)
print("=== KNN weights='distance' (k=9) ===")
print(classification_report(y_test, y_pred_dist, digits=3))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_dist)
plt.title("Matriz de confusión - weights='distance'")
plt.show()

####  Estrategias sin Re-muestreo

En este experimento analizamos dos estrategias para tratar el desbalanceo de clases en KNN **sin modificar los datos** (es decir, sin aplicar técnicas como SMOTE):

1. **Ponderación por distancia (`weights="distance"`)**
2. **Selección óptima de \( k \) mediante validación cruzada optimizando recall**



##### Ponderación por distancia

En lugar de voto uniforme:

$$
\text{voto}_i = 1
$$

se usa:

$$
\text{voto}_i = \frac{1}{d(x, x_i)}
$$

donde los vecinos más cercanos influyen más que los lejanos.


- Reduce la influencia de vecinos mayoritarios lejanos.
- Refuerza vecinos minoritarios cercanos.
- Puede mejorar el recall de la clase minoritaria sin alterar los datos.

####  Ajuste de  k optimizando recall

En lugar de elegir \( k \) maximizando accuracy, se usa:

```python
scoring="recall"
```

En datos desbalanceados:

  - Accuracy puede ser alta aunque el modelo ignore la clase minoritaria.

  - Recall penaliza falsos negativos.

  - Optimizar recall desplaza la frontera hacia la clase minoritaria.

Formalmente:

$$
\text{Recall} = \dfrac{\text{TP}}{\text{TP}+\text{FN}}
$$	​


Si la clase minoritaria es crítica (ej. diagnóstico médico), esta métrica es más adecuada.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# ============================================================
# 1) Generar datos desbalanceados 
# ============================================================
np.random.seed(42)
class_major = np.random.randn(120, 2) * 0.8 + [1, 1]
class_minor = np.random.randn(15, 2) * 0.6 + [0, 0]

X = np.vstack((class_major, class_minor))
y = np.array([0] * 120 + [1] * 15)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=422, stratify=y
)

# ============================================================
# 2) Función para dibujar frontera de decisión
# ============================================================
def plot_knn_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250),
                         np.linspace(y_min, y_max, 250))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, alpha=0.25, cmap=plt.cm.coolwarm)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.coolwarm, edgecolor="k", s=30)
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

# ============================================================
# 3) Técnica A: Ponderación por distancia (weights="distance")
# ============================================================
knn_dist = KNeighborsClassifier(n_neighbors=9, weights="distance")
knn_dist.fit(X_train, y_train)

plot_knn_decision_boundary(knn_dist, X_train, y_train, 'KNN con ponderación por distancia (k=9)')

y_pred_dist = knn_dist.predict(X_test)
print("=== KNN weights='distance' (k=9) ===")
print(classification_report(y_test, y_pred_dist, digits=3))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_dist)
plt.title("Matriz de confusión - weights='distance'")
plt.show()

# ============================================================
# 4) Técnica B: Ajuste de k (y weights) optimizando RECALL (CV)
#    - OJO: scoring="recall" asume clase positiva = 1 (como aquí)
# ============================================================
param_grid = {
    "n_neighbors": list(range(1, 32, 2)),     # k impares 1..31
    "weights": ["uniform", "distance"],
    # Si quieres, también puedes probar distintas métricas:
    # "metric": ["euclidean", "manhattan"]
}

grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    scoring="recall",   # optimiza sensibilidad de la clase 1
    cv=5
)
grid.fit(X_train, y_train)

best_knn = grid.best_estimator_
print("\n=== Mejor modelo por RECALL (CV) ===")
print("Best params:", grid.best_params_)
print("CV recall:", grid.best_score_)

plot_knn_decision_boundary(best_knn, X_train, y_train,
                           f"KNN ajustado por recall (CV): {grid.best_params_}")

y_pred_best = best_knn.predict(X_test)
print("\n=== Evaluación en TEST del mejor modelo (por recall) ===")
print(classification_report(y_test, y_pred_best, digits=3))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best)
plt.title("Matriz de confusión - Mejor KNN (scoring=recall)")
plt.show()

## Ejemplo 5: Maldición de la dimensionalidad.

In [ ]:
import numpy as np

dims = [1, 5, 10, 50, 100, 150, 200, 300, 400, 500]  # Diferentes dimensiones
n_points = 1000

for d in dims:
    points = 10*np.random.rand(n_points, d)  # Puntos en un hipercubo [0,10]^d
    dists = np.linalg.norm(points - points[0], axis=1)  # Distancia a un punto fijo
    print(f"Dimensión {d}: Distancia media = {np.mean(dists):.4f}, Desviación = {np.std(dists):.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dims = [1, 5, 10, 50, 100, 150, 200, 300, 400, 500]  # Diferentes dimensiones
n_points = 1000  # Número de puntos

# Crear un gráfico
plt.figure(figsize=(12, 8))

for d in dims:
    # Generar puntos aleatorios en un hipercubo [0, 10]^d
    points = 10 * np.random.rand(n_points, d)
    
    # Calcular la distancia entre todos los puntos
    # Usamos la norma Euclidiana entre todos los pares de puntos
    dist_matrix = np.linalg.norm(points[:, np.newaxis] - points, axis=2)
    
    # Convertimos la matriz de distancias en un vector (para graficar)
    dist_vector = dist_matrix[np.triu_indices(n_points, k=1)]  # Solo distancias por encima de la diagonal
    
    # Graficar el histograma de distancias
    plt.hist(dist_vector, bins=50, alpha=0.5, label=f'Dim={d}')

# Configuración de la gráfica
plt.title("Distribución de distancias entre puntos en diferentes dimensiones")
plt.xlabel("Distancia")
plt.ylabel("Frecuencia")
plt.legend(loc="upper right")
plt.grid(True)
plt.show()